# Use cases


## Overview
* [An alternative to the existing Apache Kafka CLI tools](#alternative)
  * [List topics](#list)
  * [Create topics](#create)
  * [Produce messages](#produce)
  * [Consume messages](#consume)
  * [Produce and consume messages using a schema](#schema)
    * [Avro](#schema_avro)
    * [Protobuf](#schema_protobuf)
    * [JSONSchema](#schema_jsonschema)
  * [Search for messages](#search)
  * [Supported serialization/seserialization types](#supported)
* [Schema Registry administration](#schema_registry)
  * [Get subjects](#get_subjects)
  * [Delete subjects](#delete_subjects)
  * [Get the latest version of a subject](#latest_version)
* [Simple stateless stream processing](#stream_processing)
  * [Copy topics](#copy)
  * [Copy + map (=map_to)](#copy_map)
  * [Copy + flatmap (=flatmap_to)](#copy_flatmap)
  * [Like a MirrorMaker](#mm)
* [Backup Kafka topics](#backup)
  * [Backing up a topic to local disk](#backup_to_local)
  * [Restoring a backed-up topic to Kafka](#restoring_to_local)
  * [Backing up a topic to S3](#backup_to_s3)
* [A bridge from Kafka to files](#bridge)
  * [Get a snapshot of a topic as a Pandas Dataframe](#topic_to_df)
  * [Write a Pandas Dataframe to a Kafka topic](#df_to_topic)
  * [Get a snapshot of a topic as an Excel file](#topic_to_xlsx)
  * [Get a snapshot of a topic as a Parquet file](#topic_to_parquet)
  * [Bring a Parquet file back to Kafka](#parquet_to_topic)
* [Debugging Kafka](#debug)
  * [Check for missing magic byte](#magic_byte)
  * [Delete records](#delete_records)
  * [Get topic watermarks](#watermarks)
  * [Collect all schemas used in a topic](#collect_schemas)


---
<a id="alternative"></a>
## An alternative to the existing Apache Kafka CLI tools

Kafi can act as an alternative to the existing Kafka CLI tools. The connection to your Kafka cluster is encapsulated in the `Cluster` object — you don't need to repeat connection details for each command.

First, make sure that you have a local Kafka running on localhost/port 9092. Then start by importing Kafi and creating a `Cluster` object:

In [ ]:
import os, sys
os.environ["KAFI_HOME"] = "../.."
sys.path.insert(1, "../..")

from kafi.kafi import *
c = Cluster("local")

<a id="list"></a>
### List topics

Kafi equivalent of `kafka-topics --bootstrap-server localhost:9092 --list`:

In [ ]:
c.ls()

<a id="create"></a>
### Create topics

Kafi equivalent of `kafka-topics --bootstrap-server localhost:9092 --topic topic_json --create`:

In [ ]:
c.touch("topic_json")

<a id="produce"></a>
### Produce messages

Producing messages works similarly to writing to a file in Python:

In [ ]:
p = c.producer("topic_json")
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

<a id="consume"></a>
### Consume messages

In [ ]:
c.cat("topic_json")

<a id="schema"></a>
### Produce and consume messages using a schema


<a id="schema_avro"></a>
#### Avro

In [ ]:
t = "topic_avro"
s = """
{
    "type": "record",
    "name": "myrecord",
    "fields": [
        {
            "name": "bla",
            "type": "int"
        }
    ]
}
"""
p = c.producer(t, value_type="avro", value_schema=s)
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

c.cat(t, value_type="avro")

<a id="schema_protobuf"></a>
#### Protobuf

In [ ]:
t = "topic_protobuf"
s = """
message the_value {
    required int32 bla = 1;
}
"""
p = c.producer(t, value_type="protobuf", value_schema=s)
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

c.cat(t, value_type="protobuf")

<a id="schema_jsonschema"></a>
#### JSONSchema

In [ ]:
t = "topic_jsonschema"
s = """
{
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "title": "myrecord",
    "properties": {
      "bla": {
        "type": "integer"
      }
    },
    "required": ["bla"],
    "additionalProperties": false
  }
"""
p = c.producer(t, value_type="jsonschema", value_schema=s)
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

c.cat(t, value_type="jsonschema")

<a id="search"></a>
### Search for messages

Use `grep` to search for messages matching a regex pattern:

In [ ]:
c.grep("topic_avro", ".*456.*", value_type="avro")

<a id="supported"></a>
### Supported serialization/deserialization types

| Type | Description |
|------|-------------|
| `bytes` | Pure bytes |
| `str` | String *(default for keys)* |
| `json` | Pure JSON *(default for values)* |
| `avro` | Avro *(requires Schema Registry)* |
| `protobuf` / `pb` | Protobuf *(requires Schema Registry)* |
| `jsonschema` / `json_sr` | JSONSchema *(requires Schema Registry)* |

Set types via:
- `key_type` / `key_schema` / `key_schema_id`
- `value_type` / `value_schema` / `value_schema_id`
- `type` — same type for both key and value

---
<a id="schema_registry"></a>
## Schema Registry administration


<a id="get_subjects"></a>
### Get Subjects

In [ ]:
c.sls()

<a id="delete_subjects"></a>
### Delete subjects

First, soft-delete:

In [ ]:
c.srm("topic_avro-value")

List subjects (soft-deleted subjects are hidden by default):

In [ ]:
c.sls()

Include soft-deleted subjects:

In [ ]:
c.sls(deleted=True)

Hard-delete the subject permanently:

In [ ]:
c.srm("topic_avro-value", permanent=True)

Verify that it's gone:

In [ ]:
c.sls(deleted=True)

<a id="latest_version"></a>
### Get the latest version of a subject

In [ ]:
c.get_latest_version("topic_jsonschema-value")

---
<a id="stream_processing"></a>
## Simple stateless stream processing


<a id="copy"></a>
### Copy Topics

Simple copy:

In [ ]:
c.cp("topic_json", c, "topic_json_copy")

Convert Protobuf topic to pure JSON:

In [ ]:
c.cp("topic_protobuf", c, "topic_avro_json_copy", source_value_type="protobuf")

Copy a pure JSON topic to an Avro topic:

In [ ]:
s = """
{
    "type": "record",
    "name": "myrecord",
    "fields": [
        {
            "name": "bla",
            "type": "int"
        }
    ]
}
"""
c.cp("topic_json", c, "topic_json_avro_copy", target_value_type="avro", target_value_schema=s)

<a id="copy_map"></a>
### Copy + map (=map_to)

Use a *single message transform* via `map_fun`:

In [ ]:
def plus_42(x):
    x["value"]["bla"] += 42
    return x

c.cp("topic_json", c, "topic_json_mapped", map_fun=plus_42)

Check the result:

In [ ]:
c.cat("topic_json_mapped")

Map also works seamlessly with schemas:

In [ ]:
c.cp("topic_protobuf", c, "topic_protobuf_json_mapped", map_fun=plus_42, source_value_type="protobuf")

<a id="copy_flatmap"></a>
### Copy + flatmap (=flatmap_to)

Use `flatmap_fun` for filtering or exploding messages:

In [ ]:
def filter_out_456(x):
    if x["value"]["bla"] == 456:
        return [x]
    else:
        return []

c.cp("topic_json", c, "topic_json_flatmapped", flatmap_fun=filter_out_456)

<a id="mm"></a>
### Like a MirrorMaker

Copy topics across clusters — input and output can be on any cluster:

In [ ]:
c1 = Cluster("local")
c2 = Cluster("local")
c1.cp("topic_json", c2, "mirrored_topic_json")

---
<a id="backup"></a>
## Backup Kafka topics


<a id="backup_to_local"></a>
### Backing up a topic to local disk

We use `type="bytes"` for a 1:1 carbon copy (no deserialization/serialization). Kafi's Kafka emulation preserves all metadata (keys, values, headers, timestamps).

In [ ]:
c = Cluster("local")
l = Local("local")
c.cp("topic_json", l, "backupped_topic_json", type="bytes")
!ls /tmp/topics/backupped_topic_json/partitions/*

<a id="restoring_to_kafka"></a>
#### Restoring a backed-up topic to Kafka

In [ ]:
l.cp("backupped_topic_json", c, "topic_json", type="bytes")

<a id="backup_to_s3"></a>
### Backing up a topic to S3

For this example to work, you need to configure `s3` correctly first (see [Full Configuration](full_configuration.ipynb))

In [ ]:
c.cp("my_topic", s3, "my_topic_backup", type="bytes")

---
<a id="bridge"></a>
## A bridge from Kafka to files

Kafi integrates with Pandas and supports these file formats: CSV, Feather, JSON, ORC, Parquet, Excel, XML.


<a id="topic_to_df"></a>
### Get a snapshot of a topic as a Pandas Dataframe

In [ ]:
df = c.topic_to_df("topic_protobuf", value_type="protobuf")
df

<a id="df_to_topic"></a>
### Write a Pandas Dataframe to a Kafka topic

In [ ]:
c.df_to_topic(df, "topic_json_from_df")
c.cat("topic_json_from_df")

<a id="topic_to_xlsx"></a>
### Get a snapshot of a topic as an Excel file

In [ ]:
l = Local("local")
c.topic_to_file("topic_json", l, "topic_json.xlsx")

<a id="topic_to_parquet"></a>
### Get a snapshot of a topic as a Parquet file

In [ ]:
l = Local("local")
c.topic_to_file("topic_json", l, "topic_json.parquet")

<a id="parquet_to_topic"></a>
### Bring a Parquet file back to Kafka

In [ ]:
l = Local("local")
l.file_to_topic("topic_json.parquet", c, "topic_json_from_parquet")

---
<a id="debug"></a>
## Debugging Kafka


<a id="magic_byte"></a>
### Check for missing magic byte

Find messages that were not properly serialized (magic byte 0 is missing):

In [ ]:
c.filter("topic_protobuf", type="bytes", filter_fun=lambda x: x["value"][0] != 0)

<a id="delete_records"></a>
### Delete records

Delete the first 100 messages of a topic:

In [ ]:
t = "topic_jsonschema"
print(c.l(t))
c.delete_records({t: {0: 2}})
c.l(t)

<a id="watermarks"></a>
### Get topic watermarks


In [ ]:
c.watermarks("topic_jsonschema")

<a id="collect_schemas"></a>
### Collect all schemas used in a topic

Collect all schema IDs in use and print the corresponding schemas from the Schema Registry:

In [ ]:
def collect_ids(acc, x):
    id = s_id(x["value"])
    acc.add(id)
    return acc

(ids, _) = c.foldl("topic_jsonschema", collect_ids, set(), type="bytes")

for id in ids:
    print(id)